# Run: tiny_cnn + MNIST, 20 rounds, WITH 7 active attackers (14%), checkpointed

Companion to `RUN_MVP_TINY_CNN_MNIST.ipynb` (the clean baseline run) --
same model, same data, same 50 clients, same 20 rounds, SAME checkpoint-
resume machinery. The one difference: 7 clients (14% -- matching
`ARCHITECTURE.md`'s own stated experimental target, contributions table
N7: "n=50 ... 7 adversaries") have their delta forced every round to a
model-replacement-scale malicious update (same construction
`RUN_ATTACKS.ipynb`'s F11 gate used to confirm V3 catches this inside a
real round), via `cheater_client_idx`/`cheater_force_delta` -- both
generalized (2026-11-16) to accept several indices at once, specifically
for this run. Everything else about each attacker's submission -- ring
signature, VRF, masking, sketching, the real range proof -- runs exactly
as it would for an honest client; see `fl/client.py`'s `_force_delta`
docstring. All 7 attackers reuse the SAME crafted delta (a simplifying
choice, noted as such -- not independently-drawn attacks per client).

**Why 7, not some other number**: `ARCHITECTURE.md` §2.2 (A2) states
the system's theoretical tolerance as `f < n/2 = 25` -- that's a ceiling
for a stress-test, not the number to validate against here. N7's own
stated target (7 of 50) is what makes this run consistent with what the
paper already commits to validating.

**Point of this run**: produce a SECOND accuracy-vs-round curve to put
next to the baseline's clean one -- "the other 43 clients converge
normally for all 20 rounds while 7 attackers (14%) are rejected by V3
every single round" is the strongest combined utility+security claim
this paper can make, and it needs this run (not just F11's 3-round/
9-client gate) to back it at the real training scale.

**Expect more aborted rounds than the baseline.** Rejected clients don't
count toward `k_min=8` eligibility, so round-availability degrades as
the malicious fraction grows -- computed exactly against this config
(`p=0.25, n=50, k_min=8`): abort probability rises from the baseline's
~4.5% (f=0) to ~9.4% at f=7. More ABORTED-round lines in this run's log
than the baseline's is the model working as predicted, not a new bug.

Run top to bottom, single Colab session (free tier), same as the
baseline notebook. If Colab disconnects, re-run from cell 1 -- it
resumes from THIS run's own checkpoint (different filename from the
baseline's), not the baseline's and not the old pre-F9-fix one.


In [ ]:
# 1. Mount Drive (checkpoint durability + MNIST cache) and clone/pull the repo.
from google.colab import drive
import os
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')

if not os.path.exists('/content/FL-ZKP'):
    !git clone https://github.com/YashasweeNarayan/FL-ZKP.git /content/FL-ZKP
else:
    !cd /content/FL-ZKP && git fetch -q origin main && git reset -q --hard origin/main
%cd /content/FL-ZKP

In [ ]:
# 2. Install the EC-dependent toolchain (rbcl for ristretto255, torch/torchvision, flwr-datasets).
!pip install -q rbcl
!pip install -q pynacl
!pip install -q flwr-datasets[vision]
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu121

**If this is the first time cell 2 has installed anything in this
session, restart the kernel now: Runtime -> Restart session.** Then
continue from cell 3 below (NOT from cell 1 -- Drive stays mounted and
the repo stays cloned across a restart, only the Python process
resets, so re-running cell 1 would be harmless but pointless).

In [ ]:
# 3. Sanity-check the install.
import sys
sys.path.insert(0, '/content/FL-ZKP')
import rbcl
assert hasattr(rbcl, "crypto_core_ristretto255_from_hash"), (
    "rbcl imported but is missing crypto_core_ristretto255_from_hash -- "
    "see crypto/ristretto.py's import fallback and FIX.md F8."
)
import torch
print("rbcl OK. torch OK, CUDA available:", torch.cuda.is_available())

In [ ]:
# 4. Load config.
import yaml
with open('configs/mvp_3day.yaml') as f:
    CFG = yaml.safe_load(f)
print(CFG)

In [ ]:
# 5. Load MNIST, stratified-IID split across clients.
# NOTE: configs/mvp_3day.yaml's datasets.mnist.source is "csv" (reads a
# Drive-cached CSV that this run's Drive doesn't have) -- that path is
# for whoever already staged the CSV. This run instead forces
# source="torchvision", which downloads MNIST directly over the network,
# no Drive dependency at all. Deliberately NOT changed in the shared
# config file, since other runs may rely on the csv path being default.
from fl.data_iid import prepare_case1

case1_data = prepare_case1(seed=CFG['seed'], source="torchvision")
print("clients:", len(case1_data['client_train']))
print("server_root size:", len(case1_data['server_root'].labels))
print("global_val size:", len(case1_data['global_val'].labels))


In [ ]:
# 6. Build the ring (one keypair per client) and the trace committee.
from crypto.ristretto import Scalar, scalar_mul_base
from crypto.dkg import generate_contribution, finalize, pk_share_for
from fl.round_loop import build_x25519_keys, precompute_pairwise_secrets

n_clients = CFG['protocol']['n_clients']
t = CFG['protocol']['threshold']
n_c = CFG['protocol']['n_key_holders']

client_sks = [Scalar.random() for _ in range(n_clients)]
ring = [scalar_mul_base(sk) for sk in client_sks]

x25519_sks, x25519_pks = build_x25519_keys(n_clients)
pairwise_secrets = precompute_pairwise_secrets(x25519_sks, x25519_pks)

key_holder_ids = list(range(1, n_c + 1))
dkg_contributions = [generate_contribution(h, t, key_holder_ids) for h in key_holder_ids]
PK_trace, key_holder_shares = finalize(dkg_contributions, key_holder_ids)
print(f"ring of {len(ring)} published, PK_trace established among {n_c} key holders (t={t})")
print(f"{n_clients} X25519 keypairs + {n_clients*(n_clients-1)} pairwise secrets precomputed")

In [ ]:
# 7. Build the model.
# tiny_cnn (~20.6K params), not resnet18: matches the model scale used
# by this paper's actual peer group (WhiteCloak NDSS'26: 19K-param CNN
# on MNIST; PQS-BFL 2025: ~21.8K-param CNN on MNIST) -- see
# fl/models.py::build_tiny_cnn's docstring for the full citation
# reasoning. ~500x fewer params than resnet18's 11.2M, which is most of
# why this run is dramatically cheaper per round.
from fl.models import build_model, flatten_state_dict

REAL_MODEL = 'tiny_cnn'
model = build_model(REAL_MODEL, num_classes=10, in_channels=1)
flat_w, model_shapes = flatten_state_dict(model)
dim = flat_w.numel()
print(REAL_MODEL, "param count:", dim)


In [ ]:
# 8. The real run: 20 rounds, checkpointed every round to Drive -- WITH
# 7 active attackers (14%, matching ARCHITECTURE.md's own N7 target) the
# whole time.
#
# Same construction F11 used to confirm V3 catches A1 inside a real
# round (FIX.md F11): each attacker's delta is forced every round to
# attacks.a1_scaling.craft_scaled_update's gamma*(w_target - w_t) output,
# built fresh from THIS model's actual initial weights (all 7 share the
# SAME crafted delta -- a simplifying choice, not independently-drawn
# attacks per client). Everything else about each attacker's submission
# (ring signature, VRF, masking, sketching, the real range proof) runs
# exactly as it would for an honest client -- see fl/client.py's
# `_force_delta` docstring for why this is NOT the same as a lying/
# forged submission. _cheat_zero_sketch does NOT also fire here
# (fl/round_loop.py's F11 fix: it's conditional on cheater_force_delta
# being None), so this is a clean, single-mechanism attack across all 7,
# not the composed one F11 found and fixed.
import numpy as np
from fl.round_loop import run_training_loop, bootstrap_norms_from_model_dim, release_gpu_memory, load_checkpoint
from attacks.a1_scaling import craft_scaled_update

REAL_ROUNDS = 20
CHEATER_INDICES = list(range(7))  # clients 0-6: 7 of 50 = 14%, matching ARCHITECTURE.md's N7 target

CKPT_DIR = '/content/drive/MyDrive/FL-ZKP/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)
# Own checkpoint filename -- distinct from the baseline run's
# (case1_tiny_cnn_r20_f9fix.pt) and the old pre-F9-fix one, so this run
# starts fresh from round 0 and never cross-resumes either.
CKPT_PATH = f'{CKPT_DIR}/case1_{REAL_MODEL}_r{REAL_ROUNDS}_f9fix_withattack7.pt'

root_norms = bootstrap_norms_from_model_dim(
    model_dim=dim,
    sketch_dim=CFG['sketch']['m'],
    sketch_seed=f"bootstrap-{REAL_MODEL}-withattack7".encode(),
    quant_bits=16,
    rng=np.random.default_rng(CFG['seed']),
)

# crafted ONCE, from this model's actual initial weights -- same
# gamma*(w_target - w_t) construction as the F11 gate, far enough from a
# random target that it should never pass V3 regardless of which round
# it's submitted in (confirmed at this exact model/config scale in F11:
# sketch-norm-squared ~378x the bound). All 7 attackers submit this SAME
# delta -- see markdown above for why that's a noted simplification.
w_target = np.random.default_rng(CFG['seed'] + 99).normal(0, 1.0, size=dim)
malicious_delta = craft_scaled_update(flat_w.numpy().astype(np.float64), w_target)
print(f"crafted malicious delta (clients {CHEATER_INDICES}), raw norm:", np.linalg.norm(malicious_delta))

print("="*70)
print(f"MALICIOUS CLIENTS IN THIS RUN: {CHEATER_INDICES} ({len(CHEATER_INDICES)} of {n_clients} = {100*len(CHEATER_INDICES)/n_clients:.0f}%)")
print("="*70)

resume_from = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
if resume_from is not None:
    print(f"found checkpoint at round {resume_from['round']} -- resuming, not restarting from round 0")

final_w_t, real_history = run_training_loop(
    model=model,
    model_dim=dim,
    client_datasets=case1_data['client_train'],
    ring=ring,
    client_sks=client_sks,
    pairwise_secrets=pairwise_secrets,
    pk_trace=PK_trace,
    server_root_norms=root_norms,
    CFG=CFG,
    rounds=REAL_ROUNDS,
    checkpoint_path=CKPT_PATH,
    resume_from=resume_from,
    cheater_client_idx=CHEATER_INDICES,
    cheater_force_delta=malicious_delta,
)
release_gpu_memory()
n_accepted = sum(1 for h in real_history if not h['aborted'])
print(f"done: {len(real_history)} rounds in history, {n_accepted} accepted (not aborted)")

print("\nper-round attacker verdict:")
attacker_ever_accepted = {}
attacker_ever_rejected = {}
for h in real_history:
    if h.get("aborted"):
        print(f"  round {h.get('round')}: ABORTED -- {h.get('reason')}")
        continue
    reasons = h.get("rejected_reasons", {})
    accepted_idxs = set(h.get("accepted_client_indices", []))
    rejected_this_round = sorted(i for i in CHEATER_INDICES if i in reasons)
    accepted_this_round = sorted(i for i in CHEATER_INDICES if i in accepted_idxs)
    for i in rejected_this_round:
        attacker_ever_rejected[i] = True
    for i in accepted_this_round:
        attacker_ever_accepted[i] = True
    print(f"  round {h.get('round')}: accepted={h.get('accepted')}, "
          f"attackers rejected={rejected_this_round}, attackers accepted={accepted_this_round}, "
          f"aggregate_exact={h.get('aggregate_exact')}")

if attacker_ever_accepted:
    print(f"\nFAIL: attacker(s) {sorted(attacker_ever_accepted)} had their scaled update accepted at least once -- a real defense failure, report this.")
elif attacker_ever_rejected:
    print(f"\nPASS: every attacker that was ever eligible ({sorted(attacker_ever_rejected)}) was rejected by V3 every time, "
          f"across all {REAL_ROUNDS} rounds, while the honest majority kept training normally.")
else:
    print("\nINCONCLUSIVE: no attacker was ever sampled/eligible in any round -- check sampling_rate/eligibility.")


## Evaluate accuracy

Works whether you let training finish, stopped it early (Ctrl-C /
interrupt), or are coming back in a fresh runtime with nothing but the
checkpoint on Drive: it loads the LATEST checkpointed weights directly
from `CKPT_PATH` (falling back to the in-memory `final_w_t` only if no
checkpoint file exists yet), so it always reports accuracy for
whatever round you actually reached, not a partially-updated `model`
object. Reusable as-is for `small_resnet` later -- every name it reads
(`REAL_MODEL`, `model`, `model_shapes`, `CKPT_PATH`, `case1_data`) is
already generic, so nothing here needs to change when you switch
backbones.


In [ ]:
import torch
from fl.models import unflatten_to_state_dict

def evaluate_accuracy(model, images, labels, batch_size=512):
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(images), batch_size):
            logits = model(images[i:i + batch_size])
            preds = logits.argmax(dim=1)
            correct += (preds == labels[i:i + batch_size]).sum().item()
    return correct / len(images)

# Load the LATEST checkpointed weights explicitly -- this is what makes
# the cell correct even if you stopped training early: `final_w_t`/
# `model` in memory only reflect whatever round your training cell
# actually got through before you interrupted it, but so does the
# checkpoint (it's written after every round), so reading from
# CKPT_PATH and reading from memory agree whenever both exist. Preferring
# the checkpoint means this cell also works standalone after a fresh
# runtime restart, with no retraining needed.
ckpt = load_checkpoint(CKPT_PATH) if os.path.exists(CKPT_PATH) else None
eval_w_t = ckpt['w_t'] if ckpt is not None else final_w_t
eval_round = ckpt['round'] if ckpt is not None else 'final (in-memory, no checkpoint found)'
model.load_state_dict(unflatten_to_state_dict(eval_w_t, model_shapes))

# Official MNIST test set (10,000 images) -- the headline number.
test_images, test_labels = case1_data['test']
test_acc = evaluate_accuracy(model, test_images, test_labels)

# global_val: a held-out slice of the TRAINING set (disjoint from every
# client's own data, see fl/splits.py) -- a secondary sanity check, not
# the headline number, since it's drawn from the same distribution the
# clients trained on rather than the official test split.
val_ds = case1_data['global_val']
val_acc = evaluate_accuracy(model, val_ds.images, val_ds.labels)

print(f"{REAL_MODEL} @ round {eval_round}: official MNIST test accuracy = {test_acc:.4f} ({len(test_images)} images)")
print(f"{REAL_MODEL} @ round {eval_round}: held-out global_val accuracy = {val_acc:.4f} ({len(val_ds.images)} images)")
